# Part 3 — Machine Learning: Predict No-Shows and Choose the Model Honestly (30 marks)

The primary clinical objective is to develop a machine learning risk-scoring model that the front desk can run the evening prior to appointments to prioritize reminder and confirmation calls. 

Because approximately only 1 in 5 bookings results in a no-show (~22%), evaluating models solely on accuracy is deceptive. Every model will be evaluated using **Precision**, **Recall**, **F1-Score**, and **ROC-AUC** alongside **Accuracy**.

### Fundamental Rules & Methodological Honesty:
1. **The Test Set is for Reporting Only:** The test set ($20\%$, $n=198$) is held out untouched and never utilized to select models, tune hyperparameters, or choose probability decision thresholds.
2. **Honest Methodological Limitations (to be reiterated in Task 4.4):**
   - *Preprocessing Leakage:* Data cleaning parameters from Task 1.3 (medians, IQR fences) were computed over all 990 records before splitting, imparting a subtle indirect influence on test observations. In production, imputers and scalers must be fit solely on training folds.
   - *Patient Overlap Across Splits:* A patient may book multiple appointments, meaning an independent random record split can place the same individual in both training and testing partitions.
   - *Temporal Disconnect:* The train/test partition is a random cross-sectional split; production systems require temporal backtesting (trained on historical bookings, deployed on future dates).

## Task 3.1 — Features, split and baseline (5 marks)

- Extract feature matrix $X$ using:
  - Numerical / ordinal features: `age`, `distance_km`, `fee_inr`, `lead_days`, `prior_no_shows`.
  - Binary mapped feature: `reminder_sent` (convert `'Yes'`/`'No'` to `1`/`0` via `.map`).
  - Categorical features: `booking_channel` and `visit_type` one-hot encoded via `pd.get_dummies(X, columns=["booking_channel", "visit_type"])`.
- Target: $y = \text{no\_show}$.
- Exclude identification/temporal metadata: `patient_id`, `appointment_id`, dates (`booked_on`, `appt_date`), `clinic_id`, `gender`, and raw `status`.
- Split dataset using `train_test_split(X, y, test_size=0.2, random_state=42)` without re-sorting or shuffling the loaded data.
- Report train/test partition sizes, number of test no-shows, and the naive baseline accuracy of predicting `"Attended"` (class 0) for all bookings.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Load clean appointments dataset from Part 1
df = pd.read_csv('appointments_clean.csv')

# Build feature matrix X and target y
candidate_features = [
    'age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows',
    'reminder_sent', 'booking_channel', 'visit_type'
]

X = df[candidate_features].copy()
X['reminder_sent'] = X['reminder_sent'].map({'Yes': 1, 'No': 0})
X = pd.get_dummies(X, columns=['booking_channel', 'visit_type'], dtype=int)
y = df['no_show']

print(f"Feature matrix X shape: {X.shape}")
print(f"Features (11 columns): {list(X.columns)}")

# Split into train and test partitions (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\nTraining set size: {len(X_train)} rows")
print(f"Test set size: {len(X_test)} rows")

# Target distributions
test_no_shows = y_test.sum()
overall_no_show_rate = y.mean()
baseline_accuracy = (y_test == 0).mean()

print(f"Test set no-shows: {test_no_shows} out of {len(y_test)} ({test_no_shows / len(y_test):.1%})")
print(f"Overall dataset no-show rate: {overall_no_show_rate:.1%}")
print(f"Baseline accuracy (always predict 'Attended'): {baseline_accuracy:.4f} ({(y_test == 0).sum()} of {len(y_test)})")

Feature matrix X shape: (990, 11)
Features (11 columns): ['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']

Training set size: 792 rows
Test set size: 198 rows
Test set no-shows: 49 out of 198 (24.7%)
Overall dataset no-show rate: 22.0%
Baseline accuracy (always predict 'Attended'): 0.7525 (149 of 198)


## Task 3.2 — Logistic regression (6 marks)

- Standardize features using `StandardScaler` fitted strictly on `X_train`.
- Fit `LogisticRegression(max_iter=1000)`.
- Report test set performance metrics: **Accuracy**, **Confusion Matrix**, **Precision**, **Recall**, **F1-Score**, and **ROC-AUC**.
- Display regression coefficients corresponding to each feature.
- Analyze the effect of `reminder_sent`:
  - Compute $e^{\beta_{\text{scaled}}}$ and $e^{\beta_{\text{scaled}} / \sigma}$.
  - Explain why the two quantities differ.
  - Formulate an observational conclusion regarding reminders and no-show odds.

In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, precision_score,
    recall_score, f1_score, roc_auc_score
)

# Standardize features (fit on training data only)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Train Logistic Regression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_s, y_train)

# Predict on test set
y_pred_lr = lr_model.predict(X_test_s)
y_prob_lr = lr_model.predict_proba(X_test_s)[:, 1]

# Calculate metrics
acc_lr = accuracy_score(y_test, y_pred_lr)
cm_lr = confusion_matrix(y_test, y_pred_lr)
prec_lr = precision_score(y_test, y_pred_lr)
rec_lr = recall_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr)
auc_lr = roc_auc_score(y_test, y_prob_lr)

print("--- Logistic Regression Test Performance ---")
print(f"Accuracy:        {acc_lr:.4f}")
print(f"Confusion Matrix:\n{cm_lr}")
print(f"Precision:       {prec_lr:.4f}")
print(f"Recall:          {rec_lr:.4f}")
print(f"F1 Score:        {f1_lr:.4f}")
print(f"ROC-AUC:         {auc_lr:.4f}")

# Display coefficients
print("\n--- Feature Coefficients ---")
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr_model.coef_[0],
    'Feature_SD': scaler.scale_
})
print(coef_df.to_string(index=False))

# Effect of reminder_sent
rem_idx = list(X.columns).index('reminder_sent')
coef_rem = lr_model.coef_[0][rem_idx]
scale_rem = scaler.scale_[rem_idx]

exp_coef_scaled = np.exp(coef_rem)
unscaled_coef = coef_rem / scale_rem
exp_coef_unscaled = np.exp(unscaled_coef)

print(f"\n--- Reminder Effect Analysis ---")
print(f"Standardized coefficient:           {coef_rem:+.4f}")
print(f"Feature Standard Deviation (scale):  {scale_rem:.4f}")
print(f"np.exp(coefficient):                {exp_coef_scaled:.4f}")
print(f"np.exp(coefficient / scale_):       {exp_coef_unscaled:.4f}")
print(f"Odds reduction (No -> Yes):         {(1 - exp_coef_unscaled):.1%}")

print("""
Explanation of Difference:
np.exp(coefficient) (0.56) represents the odds multiplier per 1 standard deviation change (~0.495) in the standardized reminder feature, whereas np.exp(coefficient / scale_) (0.31) accurately reflects the discrete unit change from 0 ('No') to 1 ('Yes'), which spans approximately 2.02 standard deviations.

Observational Conclusion:
In this fitted model, holding all other features constant, the odds of a no-show are approximately 69% lower (odds ratio ≈ 0.31) for appointments where a reminder was sent compared to those where no reminder was sent; because these data are observational, this demonstrates a strong statistical association rather than proof of causal impact.""")

--- Logistic Regression Test Performance ---
Accuracy:        0.7879
Confusion Matrix:
[[139  10]
 [ 32  17]]
Precision:       0.6296
Recall:          0.3469
F1 Score:        0.4474
ROC-AUC:         0.7889

--- Feature Coefficients ---
                Feature  Coefficient  Feature_SD
                    age    -0.253880   16.655863
            distance_km     0.253448    6.051810
                fee_inr     0.452761  184.253540
              lead_days     0.614471    8.614206
         prior_no_shows     0.649612    0.968578
          reminder_sent    -0.580669    0.494975
    booking_channel_App    -0.046804    0.486016
  booking_channel_Phone     0.215881    0.481046
booking_channel_Walk-In    -0.186364    0.435178
visit_type_Consultation     0.015658    0.492792
   visit_type_Follow-up    -0.015658    0.492792

--- Reminder Effect Analysis ---
Standardized coefficient:           -0.5807
Feature Standard Deviation (scale):  0.4950
np.exp(coefficient):                0.5595
np.exp(coef

## Task 3.3 — Decision tree and overfitting (5 marks)

- Train an unconstrained `DecisionTreeClassifier(random_state=42)` (no max depth).
- Train a regularized `DecisionTreeClassifier(max_depth=3, random_state=42)`.
- Compare training accuracy versus test accuracy for both models.
- Interpret what the performance divergence illustrates regarding variance and overfitting.

In [3]:
from sklearn.tree import DecisionTreeClassifier

# 1. Full-depth (unconstrained) tree
dt_full = DecisionTreeClassifier(random_state=42)
dt_full.fit(X_train, y_train)

train_acc_full = accuracy_score(y_train, dt_full.predict(X_train))
test_acc_full = accuracy_score(y_test, dt_full.predict(X_test))
tree_depth = dt_full.get_depth()

# 2. Pruned (max_depth=3) tree
dt_d3 = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_d3.fit(X_train, y_train)

train_acc_d3 = accuracy_score(y_train, dt_d3.predict(X_train))
test_acc_d3 = accuracy_score(y_test, dt_d3.predict(X_test))

print("--- Decision Tree Comparison ---")
print(f"Full-Depth Tree (depth={tree_depth}):")
print(f"  Training Accuracy: {train_acc_full:.4f}")
print(f"  Test Accuracy:     {test_acc_full:.4f}")

print(f"\nConstrained Tree (max_depth=3):")
print(f"  Training Accuracy: {train_acc_d3:.4f}")
print(f"  Test Accuracy:     {test_acc_d3:.4f}")

print("""
Interpretation:
The massive drop from 1.000 training accuracy to 0.6717 test accuracy in the unconstrained tree (falling well below the 0.7525 naive baseline) demonstrates severe overfitting as the tree memorized idiosyncratic noise in the training partition, whereas limiting max_depth to 3 regularized the tree to achieve 0.7778 test accuracy.""")

--- Decision Tree Comparison ---
Full-Depth Tree (depth=15):
  Training Accuracy: 1.0000
  Test Accuracy:     0.6717

Constrained Tree (max_depth=3):
  Training Accuracy: 0.8220
  Test Accuracy:     0.7778

Interpretation:
The massive drop from 1.000 training accuracy to 0.6717 test accuracy in the unconstrained tree (falling well below the 0.7525 naive baseline) demonstrates severe overfitting as the tree memorized idiosyncratic noise in the training partition, whereas limiting max_depth to 3 regularized the tree to achieve 0.7778 test accuracy.


## Task 3.4 — Random forest and a comparison table (6 marks)

- Train `RandomForestClassifier(n_estimators=100, random_state=42)`.
- Evaluate accuracy, recall, and ROC-AUC on the test set.
- Extract and display the top 3 `feature_importances_`.
- Compile a summary comparison table displaying **Accuracy**, **Recall**, and **ROC-AUC** across Logistic Regression, the Depth-3 Tree, and Random Forest.
- Provide a reporting assessment indicating which model performs best on the test set while noting that final model selection takes place strictly on training data in Task 3.5.

In [4]:
from sklearn.ensemble import RandomForestClassifier

# Fit Random Forest
rf_base = RandomForestClassifier(n_estimators=100, random_state=42)
rf_base.fit(X_train, y_train)

# Evaluate on test set
rf_train_acc = accuracy_score(y_train, rf_base.predict(X_train))
rf_test_acc = accuracy_score(y_test, rf_base.predict(X_test))
rf_test_rec = recall_score(y_test, rf_base.predict(X_test))
rf_test_auc = roc_auc_score(y_test, rf_base.predict_proba(X_test)[:, 1])

print("--- Random Forest (100 Trees) ---")
print(f"Training Accuracy: {rf_train_acc:.4f}")
print(f"Test Accuracy:     {rf_test_acc:.4f}")
print(f"Test Recall:       {rf_test_rec:.4f}")
print(f"Test ROC-AUC:      {rf_test_auc:.4f}")

# Top 3 feature importances
rf_importances = pd.Series(rf_base.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop 3 Feature Importances:")
for rank, (feat, val) in enumerate(rf_importances.head(3).items(), 1):
    print(f"  {rank}. {feat:15s}: {val:.4f}")

# Comparison Table
comparison_table = pd.DataFrame([
    {
        'Model': 'Logistic Regression',
        'Accuracy': round(acc_lr, 4),
        'Recall': round(rec_lr, 4),
        'ROC-AUC': round(auc_lr, 4)
    },
    {
        'Model': 'Decision Tree (max_depth=3)',
        'Accuracy': round(test_acc_d3, 4),
        'Recall': round(rec_lr if False else recall_score(y_test, dt_d3.predict(X_test)), 4),
        'ROC-AUC': round(roc_auc_score(y_test, dt_d3.predict_proba(X_test)[:, 1]), 4)
    },
    {
        'Model': 'Random Forest (baseline)',
        'Accuracy': round(rf_test_acc, 4),
        'Recall': round(rf_test_rec, 4),
        'ROC-AUC': round(rf_test_auc, 4)
    }
])

print("\n--- Model Comparison Table (Test Set Reporting) ---")
print(comparison_table.to_string(index=False))

print("""
Assessment:
Logistic Regression appears superior across all reporting metrics on the test partition, achieving the highest accuracy (0.7879), highest recall (0.3469), and highest ROC-AUC (0.7889); however, this table is strictly for retrospective reporting, and the formal model selection must be executed using cross-validation on training data alone in Task 3.5.""")

--- Random Forest (100 Trees) ---
Training Accuracy: 1.0000
Test Accuracy:     0.7475
Test Recall:       0.3061
Test ROC-AUC:      0.7057

Top 3 Feature Importances:
  1. lead_days      : 0.2199
  2. age            : 0.1968
  3. distance_km    : 0.1947

--- Model Comparison Table (Test Set Reporting) ---
                      Model  Accuracy  Recall  ROC-AUC
        Logistic Regression    0.7879  0.3469   0.7889
Decision Tree (max_depth=3)    0.7778  0.2245   0.6997
   Random Forest (baseline)    0.7475  0.3061   0.7057

Assessment:
Logistic Regression appears superior across all reporting metrics on the test partition, achieving the highest accuracy (0.7879), highest recall (0.3469), and highest ROC-AUC (0.7889); however, this table is strictly for retrospective reporting, and the formal model selection must be executed using cross-validation on training data alone in Task 3.5.


## Task 3.5 — Tune, choose the model and the threshold, then test once (8 marks)

- **(a) Hyperparameter Tuning:** Perform `GridSearchCV` on `RandomForestClassifier(random_state=42)` across `n_estimators: [100, 200]` and `max_depth: [3, 5, 8]` with 5-fold cross-validation evaluated on `roc_auc`.
- **(b) Honest Model Selection:** Evaluate 5-fold cross-validated ROC-AUC exclusively on the training set across Logistic Regression (using the leak-free `Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=1000))])`), the Depth-3 Tree, default Random Forest, and tuned Random Forest. Select the model with the highest cross-validated training ROC-AUC.
- **(c) Threshold Optimization:** Generate out-of-fold cross-validated probability predictions (`cross_val_predict(..., method="predict_proba")[:, 1]`) on the training partition for the winning model. Compare decision thresholds $0.50$ and $0.35$ on Precision, Recall, F1-Score, and flagged volume. Select the threshold yielding higher F1 and discuss the clinical imperative of recall.
- **(d) Final Single Evaluation:** Fit the selected model pipeline on the complete training set and evaluate it once on the unseen test set at threshold $0.35$.
- **(e) Risk Score Output:** Export `test_risk_scores.csv` containing `appointment_id`, `no_show`, `risk_prob`, and `flag` for all 198 test instances. State how operational capacity governs threshold selection in production.

In [5]:
from sklearn.model_selection import GridSearchCV, cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline

# (a) Tune Random Forest via GridSearchCV on training set
rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 8]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    rf_param_grid,
    cv=5,
    scoring='roc_auc'
)
rf_grid.fit(X_train, y_train)

print("--- (a) Random Forest Grid Search Results ---")
print(f"Best Hyperparameters: {rf_grid.best_params_}")
print(f"Best CV ROC-AUC:      {rf_grid.best_score_:.4f}")

# (b) Model Selection on Training Set (5-fold CV ROC-AUC)
# Supplied Pipeline for Logistic Regression (re-scales inside every fold)
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

cv_auc_lr = cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean()
cv_auc_dt3 = cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=42), X_train, y_train, cv=5, scoring="roc_auc").mean()
cv_auc_rf_base = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42), X_train, y_train, cv=5, scoring="roc_auc").mean()
cv_auc_rf_tuned = rf_grid.best_score_

print("\n--- (b) 5-Fold Cross-Validated Training ROC-AUC ---")
print(f"  Logistic Regression (Pipeline):  {cv_auc_lr:.4f}")
print(f"  Decision Tree (depth=3):         {cv_auc_dt3:.4f}")
print(f"  Random Forest (default):         {cv_auc_rf_base:.4f}")
print(f"  Random Forest (tuned):           {cv_auc_rf_tuned:.4f}")

# Select winning model based strictly on CV score
model_scores = {
    'Logistic Regression': cv_auc_lr,
    'Decision Tree (depth=3)': cv_auc_dt3,
    'Random Forest (default)': cv_auc_rf_base,
    'Random Forest (tuned)': cv_auc_rf_tuned
}
winner_name = max(model_scores, key=model_scores.get)
print(f"Winning Model: {winner_name} (highest training CV ROC-AUC)")

# (c) Threshold Selection on Training Set (Out-of-Fold Probabilities)
oof_probs = cross_val_predict(lr_pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]

print("\n--- (c) Out-of-Fold Threshold Tuning on Training Set ---")
for threshold in [0.5, 0.35]:
    flags = (oof_probs >= threshold).astype(int)
    p = precision_score(y_train, flags)
    r = recall_score(y_train, flags)
    f = f1_score(y_train, flags)
    count = flags.sum()
    print(f"Threshold {threshold:4.2f} -> Precision: {p:.4f}, Recall: {r:.4f}, F1: {f:.4f}, Flagged: {count:3d}")

print("""
Threshold Choice Rationale:
Threshold 0.35 is chosen because it produces a substantially higher F1-score (0.4892 vs 0.3471) and nearly doubles recall (0.4675 vs 0.2485); in an outpatient clinic, recall is paramount because an undetected no-show leaves an expensive consultation slot wasted, whereas a false alarm costs only an extra one-minute confirmation phone call.""")

# (d) Fit winning model on full training set and evaluate ONCE on test set
lr_pipe.fit(X_train, y_train)
test_probs = lr_pipe.predict_proba(X_test)[:, 1]
test_flags = (test_probs >= 0.35).astype(int)

final_precision = precision_score(y_test, test_flags)
final_recall = recall_score(y_test, test_flags)
final_flagged = test_flags.sum()
final_roc_auc = roc_auc_score(y_test, test_probs)

print("\n--- (d) Final Test Set Evaluation (Threshold = 0.35) ---")
print(f"Precision:      {final_precision:.4f}")
print(f"Recall:         {final_recall:.4f}")
print(f"Number Flagged: {final_flagged} of {len(y_test)}")
print(f"ROC-AUC:        {final_roc_auc:.4f}")

# (e) Save test_risk_scores.csv
test_risk_scores = pd.DataFrame({
    'appointment_id': df.loc[X_test.index, 'appointment_id'].values,
    'no_show': y_test.values,
    'risk_prob': np.round(test_probs, 4),
    'flag': test_flags
})

test_risk_scores.to_csv('test_risk_scores.csv', index=False)
print(f"\n--- (e) Exported test_risk_scores.csv ---")
print(f"File shape: {test_risk_scores.shape}")
print(f"Flagged appointments (flag = 1): {test_risk_scores['flag'].sum()}")
display(test_risk_scores.head(10))

print("""
Operational Threshold Selection:
In a real clinical environment, management would set the decision threshold based directly on the front desk's staffing capacity—specifically balancing the finite volume of phone calls staff can make in an evening against the clinical and financial cost of an unfilled doctor slot.""")

--- (a) Random Forest Grid Search Results ---
Best Hyperparameters: {'max_depth': 3, 'n_estimators': 200}
Best CV ROC-AUC:      0.7673



--- (b) 5-Fold Cross-Validated Training ROC-AUC ---
  Logistic Regression (Pipeline):  0.7737
  Decision Tree (depth=3):         0.6784
  Random Forest (default):         0.7466
  Random Forest (tuned):           0.7673
Winning Model: Logistic Regression (highest training CV ROC-AUC)

--- (c) Out-of-Fold Threshold Tuning on Training Set ---
Threshold 0.50 -> Precision: 0.5753, Recall: 0.2485, F1: 0.3471, Flagged:  73
Threshold 0.35 -> Precision: 0.5130, Recall: 0.4675, F1: 0.4892, Flagged: 154

Threshold Choice Rationale:
Threshold 0.35 is chosen because it produces a substantially higher F1-score (0.4892 vs 0.3471) and nearly doubles recall (0.4675 vs 0.2485); in an outpatient clinic, recall is paramount because an undetected no-show leaves an expensive consultation slot wasted, whereas a false alarm costs only an extra one-minute confirmation phone call.

--- (d) Final Test Set Evaluation (Threshold = 0.35) ---
Precision:      0.5714
Recall:         0.5714
Number Flagged: 49 of 198


,appointment_id,no_show,risk_prob,flag
0,A0217,0,0.0216,0
1,A0336,0,0.1509,0
2,A0508,0,0.5187,1
3,A0314,0,0.1913,0
4,A0089,0,0.1677,0
5,A0543,1,0.3200,0
6,A0284,0,0.3384,0
7,A0109,1,0.3670,1
8,A0060,0,0.1050,0
9,A0513,0,0.1733,0



Operational Threshold Selection:
In a real clinical environment, management would set the decision threshold based directly on the front desk's staffing capacity—specifically balancing the finite volume of phone calls staff can make in an evening against the clinical and financial cost of an unfilled doctor slot.


## Acceptance Criteria Verification (Part 3)

Validating all metrics against formal Part 3 acceptance criteria:
- **3.1**: X has 11 columns; train 792 rows, test 198 rows; 49 test no-shows; baseline accuracy 0.7525.
- **3.2**: Accuracy ≈ 0.788, precision ≈ 0.63, recall ≈ 0.35, F1 ≈ 0.45, ROC-AUC ≈ 0.789; CM = [[139, 10], [32, 17]]; reminder_sent coef ≈ -0.58, exp(coef) ≈ 0.56, exp(coef/scale) ≈ 0.31 (~69% lower odds).
- **3.3**: Full-depth tree train 1.00, test ≈ 0.67 (depth ≈ 15); depth-3 tree train ≈ 0.82, test ≈ 0.78.
- **3.4**: RF train 1.00, test acc ≈ 0.75, recall ≈ 0.31, ROC-AUC ≈ 0.71; lead_days among top 3; Logistic Regression highest ROC-AUC.
- **3.5**: RF grid search best CV ROC-AUC ≈ 0.77; CV training ROC-AUC: LR ≈ 0.77, DT3 ≈ 0.68, RF ≈ 0.75, RF tuned ≈ 0.77; out-of-fold threshold 0.5 flagged ≈ 73, threshold 0.35 flagged ≈ 154 (0.35 chosen); test evaluation at 0.35 has precision ≈ 0.57, recall ≈ 0.57, flagged = 49, ROC-AUC ≈ 0.79; `test_risk_scores.csv` has 198 rows and 49 flagged.

In [6]:
# Formal assertion checks against acceptance criteria
# 3.1
assert X.shape[1] == 11, "3.1 Failed: X column count != 11"
assert len(X_train) == 792 and len(X_test) == 198, "3.1 Failed: train/test split size"
assert test_no_shows == 49, "3.1 Failed: test no-shows != 49"
assert round(baseline_accuracy, 4) == 0.7525, "3.1 Failed: baseline accuracy != 0.7525"

# 3.2
assert abs(acc_lr - 0.788) < 0.03, "3.2 Failed: accuracy tolerance"
assert abs(prec_lr - 0.63) < 0.03, "3.2 Failed: precision tolerance"
assert abs(rec_lr - 0.35) < 0.06, "3.2 Failed: recall tolerance"
assert abs(auc_lr - 0.789) < 0.04, "3.2 Failed: ROC-AUC tolerance"
assert (cm_lr == np.array([[139, 10], [32, 17]])).all(), "3.2 Failed: confusion matrix"
assert round(exp_coef_scaled, 2) == 0.56, "3.2 Failed: exp(coef) != 0.56"
assert round(exp_coef_unscaled, 2) == 0.31, "3.2 Failed: exp(coef/scale) != 0.31"

# 3.3
assert train_acc_full == 1.00, "3.3 Failed: full tree train acc != 1.00"
assert abs(test_acc_full - 0.67) < 0.03, "3.3 Failed: full tree test acc != 0.67"
assert abs(train_acc_d3 - 0.82) < 0.03, "3.3 Failed: depth-3 train acc != 0.82"
assert abs(test_acc_d3 - 0.78) < 0.03, "3.3 Failed: depth-3 test acc != 0.78"

# 3.4
assert rf_train_acc == 1.00, "3.4 Failed: RF train acc != 1.00"
assert abs(rf_test_acc - 0.75) < 0.03, "3.4 Failed: RF test acc != 0.75"
assert abs(rf_test_auc - 0.71) < 0.04, "3.4 Failed: RF test ROC-AUC != 0.71"
assert 'lead_days' in rf_importances.head(3).index, "3.4 Failed: lead_days not in top 3"

# 3.5
assert abs(rf_grid.best_score_ - 0.77) < 0.03, "3.5 Failed: RF best_score_ != 0.77"
assert abs(cv_auc_lr - 0.77) < 0.03, "3.5 Failed: LR CV ROC-AUC != 0.77"
assert abs(final_precision - 0.57) < 0.03, "3.5 Failed: final test precision != 0.57"
assert abs(final_recall - 0.57) < 0.05, "3.5 Failed: final test recall != 0.57"
assert final_flagged == 49, "3.5 Failed: final flagged != 49"
assert abs(final_roc_auc - 0.79) < 0.03, "3.5 Failed: final test ROC-AUC != 0.79"

assert len(test_risk_scores) == 198, "3.5 Failed: test_risk_scores row count != 198"
assert test_risk_scores['flag'].sum() == final_flagged, "3.5 Failed: flag count mismatch"

print("ALL PART 3 ACCEPTANCE CRITERIA PASSED SUCCESSFULLY! ✓")

ALL PART 3 ACCEPTANCE CRITERIA PASSED SUCCESSFULLY! ✓
